# V2 · Figure-Type Classification: text vs vision vs fusion (SPIQA)
Classify scientific figures into **plot, schematic, photo, table, other**, on the same SPIQA figures with three inputs:

| Run | Input | Track |
|---|---|---|
| `vit_base` | image | Vision (main model) |
| `scibert_caption` | caption text | NLP |
| `fusion_vit_scibert` | image + caption, learned gate | Multimodal fusion |
| `late_fusion_avg` | averaged probabilities (no training, built in the report) | fusion baseline |
| `vit_linear_probe`, `resnet50` | image | vision baselines |

Each section can be re-run after a disconnect: finished seeds are skipped automatically.

## 0. Setup
Runtime → Change runtime type → **T4 GPU**. Add `HF_TOKEN` in the Secrets panel (key icon) with notebook access on.
Edit the two settings below once.

In [ ]:
REPO_URL = "https://github.com/mkavinvelavan/ResearchLens.git"
HF_USERNAME = "YOUR_HF_USERNAME"                                          # <- change
DRIVE_ROOT = "/content/drive/MyDrive/ResearchLens"                        # checkpoints + results live here

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)

if not os.path.exists("/content/researchlens"):
    !git clone -q {REPO_URL} /content/researchlens
else:
    !git -C /content/researchlens pull -q
%cd /content/researchlens
!pip install -q -r requirements-colab.txt

os.environ["HF_USERNAME"] = HF_USERNAME
os.environ["RL_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TQDM_DISABLE"] = "1"   # plain log lines instead of progress bars
# Colab Secrets are only readable here in the notebook, so pass the token on to the training scripts.
from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN found")
except Exception as e:
    print("WARNING: HF_TOKEN secret not found / notebook access off -> models will not be pushed to the Hub.", e)
!nvidia-smi --query-gpu=name,memory.total --format=csv

import subprocess, sys
def run(module, config, run_name, seed, extra=""):
    # Train one (run, seed) unless it already finished (metrics.json on Drive): safe to re-run after a disconnect.
    import yaml
    tag = yaml.safe_load(open(config))["model_tag"]
    done = f"{DRIVE_ROOT}/results/{tag}/{run_name}/seed_{seed}/metrics.json"
    if os.path.exists(done):
        print(f"skip {run_name} seed {seed} (finished)"); return
    if "--push_to_hub" in extra and not os.environ.get("HF_TOKEN"):
        extra = extra.replace("--push_to_hub", "")
    cmd = f"{sys.executable} -m {module} --config {config} --run_name {run_name} --seed {seed} {extra}"
    print(">>>", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:            # stream the training log live into the notebook
        print(line, end="", flush=True)
        tail = (tail + [line])[-40:]
    if p.wait() != 0:
        raise RuntimeError(f"Training failed: {run_name} seed {seed}. Last lines:\n" + "".join(tail))

## 1. Dataset (SPIQA → 5 figure types)
First time: downloads SPIQA from the Hub, samples up to 3,000 training figures per class, shrinks them to ≤512 px
and caches a zip on Drive. Later sessions just unzip the cache (about a minute).

In [ ]:
CACHE = f"{DRIVE_ROOT}/data/spiqa_figs.zip"
if os.path.exists("/content/data/spiqa_figs/manifest.csv"):
    print("dataset ready")
elif os.path.exists(CACHE):
    !mkdir -p /content/data && unzip -q -o "{CACHE}" -d /content/data
else:
    !python -m researchlens.vision.prepare_spiqa_figures --out_dir /content/data/spiqa_figs --max_per_class 3000
    os.makedirs(f"{DRIVE_ROOT}/data", exist_ok=True)
    !cd /content/data && zip -q -r "{CACHE}" spiqa_figs
!head -3 /content/data/spiqa_figs/manifest.csv

## 2. Vision: ViT on images (3 seeds; seed 42 pushed to the Hub)

In [ ]:
SEEDS = [42, 43, 44]
for s in SEEDS:
    run("researchlens.vision.train_figcls", "configs/v2_figcls.yaml", "vit_base", s, "--push_to_hub" if s == 42 else "")

## 3. NLP: SciBERT on captions only (3 seeds)

In [ ]:
for s in SEEDS:
    run("researchlens.nlp.train_caption_cls", "configs/v2_caption.yaml", "scibert_caption", s, "--push_to_hub" if s == 42 else "")

## 4. Fusion: ViT + SciBERT with a learned gate (3 seeds)

In [ ]:
for s in SEEDS:
    run("researchlens.fusion.train_figcls_fusion", "configs/v2_fusion.yaml", "fusion_vit_scibert", s, "--push_to_hub" if s == 42 else "")

## 5. Vision baselines: ViT linear probe + ResNet-50 CNN

In [ ]:
for s in SEEDS:
    run("researchlens.vision.train_figcls", "configs/v2_figcls.yaml", "vit_linear_probe", s, "--freeze_backbone")
for s in SEEDS:
    run("researchlens.vision.train_figcls", "configs/v2_figcls.yaml", "resnet50", s, "--model_name microsoft/resnet-50")

## 6. ViT attention maps + misclassified examples (needs the images, so run in this session)

In [ ]:
R = f"{DRIVE_ROOT}/results/v2_figcls/vit_base/seed_42"
M = f"{DRIVE_ROOT}/models/v2_figcls/vit_base/seed_42"
!python -m researchlens.vision.explain --results_dir {R} --model_dir {M} --data_dir /content/data/spiqa_figs

## 7. Report: all graphs, tables and the significance test

In [ ]:
import pandas as pd
from researchlens.evaluation.reports import figcls_report
REPORT_DIR = f"{DRIVE_ROOT}/results/v2_figcls/report"
display(figcls_report(f"{DRIVE_ROOT}/results"))
display(pd.read_csv(f"{REPORT_DIR}/significance_mcnemar.csv"))

from pathlib import Path
from IPython.display import Image, display
for f in sorted(Path(REPORT_DIR).glob("*.png")):
    print(f.name); display(Image(str(f), width=650))

for f in sorted(Path(R, "figures").glob("*.png")):
    print(f.name); display(Image(str(f), width=800))